In [ ]:
import torch

In [ ]:
class AttentionLayer(torch.nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self, query, key, value):
    outputs = torch.matmul(query, key.transpose(-2, -1))
    outputs = outputs / (key.size(-1)**0.5)
    outputs = torch.nn.functional.softmax(outputs, dim=-1)
    return torch.matmul(outputs, value)

# # 1. Instantiate the single component
# attn = AttentionLayer()

# # 2. Fabricate dummy inputs with exact expected shapes [B, H, T, D]
# dummy_q = torch.randn(4, 8, 32, 64)
# dummy_k = torch.randn(4, 8, 32, 64)
# dummy_v = torch.randn(4, 8, 32, 64)

# # 3. Execute just this layer
# out = attn(dummy_q, dummy_k, dummy_v)

# # 4. Assert the exact output shape
# assert out.shape == (4, 8, 32, 64), f"Shape mismatch: {out.shape}"

# # Verify probabilities sum to 1.0 along the last dimension
# prob_sums = outputs.sum(dim=-1)
# assert torch.allclose(prob_sums, torch.ones_like(prob_sums))

class MultiHeadAttentionLayer(torch.nn.Module):
  def __init__(self, h, C, attentionLayer):
    super().__init__()

    self.h = h
    self.C = C
    self.head_dim = C // h

    self.queryLayer = torch.nn.Linear(C, C)
    self.keyLayer = torch.nn.Linear(C, C)
    self.valueLayer = torch.nn.Linear(C,C)

    self.attention = attentionLayer

    self.outputLayer = torch.nn.Linear(C,C)

  def forward(self, query, key, value):
    B = query.shape[0]
    T = query.shape[1]

    Q = self.queryLayer(query)
    K = self.keyLayer(key)
    V = self.valueLayer(value)

    Q_splits = Q.view(B, T, self.h, self.head_dim).transpose(1,2)
    K_splits = K.view(B, T, self.h, self.head_dim).transpose(1,2)
    V_splits = V.view(B, T, self.h, self.head_dim).transpose(1,2)

    output = self.attention(Q_splits, K_splits, V_splits)
    output = output.transpose(1,2)
    output = output.contigous().view(B, T, self.C)

    return self.outputLayer(output)


  
